In [ ]:
!nvidia-smi
!git clone -q https://github.com/Amarnath10i/research-on-hyperspectral--image-fusion.git /kaggle/working/repo && cd /kaggle/working/repo && git checkout -q 548c3df && git log --oneline -1
!python -c "import torch, h5py; print(torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# 1) PUFormer - main model
import subprocess, sys
p = subprocess.Popen('python train.py --mat /kaggle/input --cache /tmp/chikusei_crop.npy --model puformer --width 48 --stages 3 --bs 8 --hours 9.5 --out /kaggle/working/out/puformer', shell=True, cwd='/kaggle/working/repo/chikusei_sota',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end='', flush=True)
assert p.wait() == 0, 'command failed'


In [ ]:
# 2) gap evaluation: consistency / blur+SRF mismatch with operator swap / noise
import subprocess, sys
p = subprocess.Popen('python eval_gaps.py --mat /kaggle/input --cache /tmp/chikusei_crop.npy --model puformer --width 48 --stages 3 --ckpt /kaggle/working/out/puformer/best_ema.pt --out /kaggle/working/out/puformer', shell=True, cwd='/kaggle/working/repo/chikusei_sota',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end='', flush=True)
assert p.wait() == 0, 'command failed'


In [ ]:
# 3) SSRNet anchor under the identical protocol
import subprocess, sys
p = subprocess.Popen('python train.py --mat /kaggle/input --cache /tmp/chikusei_crop.npy --model ssrnet --bs 16 --lr 1e-3 --iters 30000 --hours 0.5 --eval_every 2000 --out /kaggle/working/out/ssrnet', shell=True, cwd='/kaggle/working/repo/chikusei_sota',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end='', flush=True)
assert p.wait() == 0, 'command failed'


In [ ]:
import json, glob
for f in sorted(glob.glob('/kaggle/working/out/*/results.json')):
    r = json.load(open(f)); print(f, r['params_M'], {k: round(v, 4) for k, v in r['test'].items()})
!rm -rf /kaggle/working/repo /kaggle/working/out/*/last.pt